# Web Requests and Scraping

Almost every interesting data source on the internet is accessible over HTTP: REST APIs, public websites, government data portals. `requests` gives you a clean, readable API for making HTTP calls, and `BeautifulSoup` turns raw HTML into a navigable tree you can query like a database. Together they let you pull structured data from the web in a few lines of Python.

**What's inside:** GET and POST requests, query parameters, custom headers, timeout and error handling, sessions, HTML parsing, CSS selectors, tree navigation, and practical scraping patterns including pagination.

**Learn more:** [requests documentation](https://docs.python-requests.org) · [BeautifulSoup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)

## Setup

In [ ]:
%pip install requests beautifulsoup4

## 1. requests

### 1.1 GET request

The response object contains the status code, headers, and body.

In [ ]:
import requests

response = requests.get('https://httpbin.org/get')
response.status_code

### 1.2 Response attributes

In [ ]:
print(response.status_code)               # HTTP status
print(response.headers['Content-Type'])   # response headers
print(len(response.text))                 # body as a string

In [ ]:
# .json() parses the response body as JSON
response.json()['url']

### 1.3 Query parameters

Pass a dict to `params`; requests handles URL encoding automatically.

In [ ]:
response = requests.get('https://httpbin.org/get', params={'q': 'python', 'page': 1})
response.url   # shows the encoded URL

In [ ]:
# the params appear in the response body under 'args'
response.json()['args']

### 1.4 POST with a JSON body

In [ ]:
response = requests.post('https://httpbin.org/post', json={'name': 'Alice', 'score': 95})
response.json()['json']   # echoes back the JSON body we sent

### 1.5 Custom headers

In [ ]:
headers = {'User-Agent': 'my-scraper/1.0'}
response = requests.get('https://httpbin.org/headers', headers=headers)
response.json()['headers']['User-Agent']

### 1.6 Timeout and error handling

`raise_for_status()` raises an `HTTPError` for 4xx and 5xx responses.

In [ ]:
try:
    response = requests.get('https://httpbin.org/status/404', timeout=5)
    response.raise_for_status()
except requests.exceptions.HTTPError as e:
    print(e)

In [ ]:
# timeout= prevents hanging on slow servers
try:
    requests.get('https://httpbin.org/delay/10', timeout=2)
except requests.exceptions.Timeout:
    print('request timed out')

### 1.7 Session: reuse headers and cookies

A `Session` persists headers, cookies, and connection pooling across requests.

In [ ]:
session = requests.Session()
session.headers.update({'User-Agent': 'my-scraper/1.0'})

r = session.get('https://httpbin.org/headers')
r.json()['headers']['User-Agent']   # header applied automatically

## 2. BeautifulSoup

### 2.1 Parsing HTML

`html.parser` is the built-in parser (no extra dependencies).

In [ ]:
from bs4 import BeautifulSoup

html = '''
<html>
  <body>
    <h1>Welcome</h1>
    <p class="intro">Hello, <strong>world</strong>!</p>
    <ul>
      <li><a href="/one">One</a></li>
      <li><a href="/two">Two</a></li>
      <li><a href="/three">Three</a></li>
    </ul>
  </body>
</html>
'''

soup = BeautifulSoup(html, 'html.parser')

### 2.2 Finding elements

`find()` returns the first match; `find_all()` returns a list of all matches.

In [ ]:
soup.find('h1').text

In [ ]:
[li.text for li in soup.find_all('li')]

In [ ]:
# filter by attribute
soup.find('p', class_='intro').text

### 2.3 CSS selectors

`select()` uses CSS selector syntax and returns a list; `select_one()` returns the first match.

In [ ]:
soup.select_one('p.intro strong').text

In [ ]:
# all links inside list items
[a.text for a in soup.select('li a')]

### 2.4 Extracting attributes

Use `tag['attr']` or the safer `tag.get('attr')` which returns `None` if absent.

In [ ]:
[a['href'] for a in soup.find_all('a')]

In [ ]:
# .get() avoids KeyError if the attribute is missing
[a.get('title', 'no title') for a in soup.find_all('a')]

### 2.5 Navigating the tree

In [ ]:
p = soup.find('p')
print(p.parent.name)     # parent tag
print(p.strong.text)     # child by tag name

In [ ]:
ul = soup.find('ul')
[li.text for li in ul.children if li.name == 'li']

### 2.6 Extracting all text

`get_text()` strips all tags and returns the plain text content.

In [ ]:
soup.get_text(separator=' ', strip=True)

## 3. Fetching and Parsing Live Pages

The examples below use [books.toscrape.com](https://books.toscrape.com), a site built specifically for scraping practice.

### 3.1 Fetch a page and parse it

In [ ]:
response = requests.get('https://books.toscrape.com', timeout=10)
soup = BeautifulSoup(response.text, 'html.parser')
soup.title.text

### 3.2 Extract all links

In [ ]:
# href=True filters out anchors with no href attribute
links = [a['href'] for a in soup.find_all('a', href=True)]
links[:5]

### 3.3 Scrape structured data

Extract book titles and prices from the catalogue listing.

In [ ]:
books = []
for article in soup.select('article.product_pod')[:5]:
    title = article.h3.a['title']
    price = article.select_one('p.price_color').text.strip()
    rating = article.p['class'][1]   # 'One', 'Two', ... 'Five'
    books.append({'title': title, 'price': price, 'rating': rating})

books

### 3.4 Scrape a table

HTML tables map directly to rows and columns.

In [ ]:
# fetch the detail page of the first book
first_url = 'https://books.toscrape.com/' + soup.select_one('article.product_pod h3 a')['href']
detail = BeautifulSoup(requests.get(first_url, timeout=10).text, 'html.parser')

# the product information table
rows = detail.select('table.table tr')
{row.th.text: row.td.text for row in rows}

## 4. Practical Patterns

### 4.1 Reusable fetch-and-parse helper

In [ ]:
def fetch(url: str, session: requests.Session | None = None) -> BeautifulSoup:
    requester = session or requests
    response = requester.get(url, timeout=10)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')

fetch('https://books.toscrape.com').title.text

### 4.2 Polite scraping: pause between requests

Adding a short delay avoids hammering the server and getting rate-limited or blocked.

In [ ]:
import time

urls = [
    'https://books.toscrape.com/catalogue/page-1.html',
    'https://books.toscrape.com/catalogue/page-2.html',
]

for url in urls:
    soup = fetch(url)
    count = len(soup.select('article.product_pod'))
    print(f'{count} books on {url.split("/")[-1]}')
    time.sleep(1)   # 1 second between requests

### 4.3 Paginate through results

In [ ]:
base = 'https://books.toscrape.com/catalogue/page-{}.html'
all_titles = []

for page in range(1, 4):   # first 3 pages
    soup = fetch(base.format(page))
    titles = [a.h3.a['title'] for a in soup.select('article.product_pod')]
    all_titles.extend(titles)
    time.sleep(1)

print(f'{len(all_titles)} books collected')
all_titles[:5]